# Step 2: RAG evaluation
Final Step 2 pipeline = `rag_pipeline.rag_answer` (hybrid retrieval + date table + answer rules), 50/52 correct.

Results are cached per **mode and model** in `outputs/cache_<mode>_<model>.json`. The 50/52 run is cached as mode
`rag_v4` with `openai/gpt-oss-120b`, so to reproduce the report numbers without new LLM calls set
`LLM_MODEL=openai/gpt-oss-120b` in `.env` (same for the Step 1 `baseline`).

In [1]:
import pandas as pd
import numpy as np
from config import OUT_DIR, TOP_K, LLM_MODEL
from eval_cache import run_cached, score_cached, summary
from rag_pipeline import retrieve, rag_answer, chunk_kb
print("model:", LLM_MODEL, "| TOP_K:", TOP_K)

model: nvidia/nemotron-3.5-super-vl-120b-a12b | TOP_K: 26


In [2]:
chunks = pd.DataFrame(chunk_kb())
print(len(chunks), "chunks")
print(chunks.groupby(["doc_type", "status"]).size())

154 chunks
doc_type     status    
agent_guide  current       13
catalog      current       70
faq          current        9
help         current        9
marketing    current        1
meta         current        4
policy       current       46
             superseded     2
dtype: int64


In [3]:
# what retrieval returns for a customer-style question
for h in retrieve("nobody came to collect my return", k=6):
    print(h["score"], h["id"], h["status"], h["title"])

/Users/purvidharmana/Desktop/DataScience/VoiceOps/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 11:09:27.711000 14457 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14340.36it/s]


0.0457 RVP-3 current Item rejected at the door
0.0444 RET-2 current Return windows by category
0.0437 RET-3 current Change to electronics returns
0.0423 FAQ-2 current Can I return something I bought?
0.0411 RET-6 current Reverse pickup and fees
0.0404 FAQ-1 current How long do refunds take?


In [4]:
MODE = "rag_v4"                      # the 50/52 run; cached questions are not sent to the LLM again
run_cached(MODE, rag_answer)

rag_v4: 0 cached, 52 to run
  PQ0002: success answer
  PQ0003: success answer
  PQ0004: success escalate
  PQ0005: success answer
  PQ0006: success answer
  PQ0008: success answer
  PQ0009: success escalate
  PQ0010: success answer
  PQ0013: success answer
  PQ0014: success answer
  PQ0015: success escalate
  PQ0016: success answer
  PQ0018: success clarify
  PQ0019: success answer
  PQ0020: success answer
  PQ0021: success answer
  PQ0022: success answer
  PQ0023: success escalate
  PQ0024: success answer
  PQ0025: success answer
  PQ0026: success answer
  PQ0027: success answer
  PQ0028: success answer
  PQ0029: success answer
  PQ0030: success escalate
  PQ0031: success answer
  PQ0032: success answer
  PQ0033: success answer
  PQ0034: success answer
  PQ0035: success answer
  PQ0036: success answer
  PQ0037: success escalate
  PQ0039: success answer
  PQ0040: success answer
  PQ0041: success answer
  PQ0042: success answer
  PQ0044: success answer
  PQ0045: success answer
  PQ0046:

{'PQ0002': {'llm_ok': True,
  'json_ok': True,
  'error': None,
  'decision': 'answer',
  'reply': 'Haan, SKU ELE-0012 par ZentroMart Assured badge hai. Is par 12 mahine ki ZentroMart Assured warranty delivery date se milti hai. Claim karne ke liye app mein My Orders mein jayein aur claim raise karein; ZentroMart pickup arrange karega aur repair ya replacement karega.',
  'used_sections': ['products.csv:ELE-0012', 'WAR-2'],
  'retrieved': ['products.csv:ELE-0012',
   'WAR-2',
   'products.csv:ELE-0023',
   'MKS-3',
   'WAR-1',
   'WAR-3',
   'DWM-4',
   'DEL-4',
   'PLS-1',
   'RET-1',
   'RET-7',
   'MKT-1',
   'REF-1',
   'PLS-2',
   'ESC-1',
   'FAQH-4',
   'STY-1',
   'DEL-5',
   'PAY-4',
   'MKS-1',
   'MKS-2',
   'REF-3',
   'DEL-2',
   'PLS-3',
   'FAQH-2',
   'RET-6'],
  'cited_raw': ['products.csv:ELE-0012', 'WAR-2'],
  'context_chars': 11422,
  'llm_status': 'success'},
 'PQ0003': {'llm_ok': True,
  'json_ok': True,
  'error': None,
  'decision': 'answer',
  'reply': 'Refunds

In [5]:
base, rag = score_cached("baseline"), score_cached(MODE)
table = pd.DataFrame({"Step 1 (no docs)": summary(base), "Step 2 (RAG)": summary(rag)})
table.to_csv(OUT_DIR / "step1_vs_step2.csv")
table

,Step 1 (no docs),Step 2 (RAG)
scored,52,52
correct,12 / 52 (23.1%),52 / 52 (100.0%)
decision accuracy,25.0%,100.0%
avg fact recall,30.8%,100.0%
violations,0,0
recall@k,NaN,0.933
hit@k,NaN,1.0
mrr,NaN,0.641
cite_precision,NaN,0.748


In [6]:
fail = rag[~rag.correct].copy()
fail["cause"] = np.where(fail["hit@k"] == 0, "retrieval miss",
                np.where(~fail.decision_ok, "wrong decision", "facts missing / violation"))
print(fail.cause.value_counts())
for _, r in fail.iterrows():
    print(f"{r.id} [{r.language}] {r.cause} | expected={r.expected}, got={r.decision}")
    print(f"  missing: {r.missing}\n  reply:   {r.reply}\n")

Series([], Name: count, dtype: int64)
